# ML Lab - EDA Pipeline (Exp1)

## i) Loading the dataset ii) Exploratory Data Analysis and Visualization iii) Data Preprocessing

#### Importing Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import LabelEncoder
#import torch
#import torch.nn as nn
#import torch.optim as optim
#from torch.utils.data import DataLoader, TensorDataset
#import tensorflow as tf
#from tensorflow import keras
#from tensorflow.keras import layers, models, Sequential

#### Dataset Retreival

In [ ]:
file = input("Enter the path: ")
if file.endswith(".csv"):
    df=pd.read_csv(file)
elif file.endswith(".xlsx"):
    df=pd.read_excel(file)
#elif file.endswith(".data"):
    #df = pd.read_csv(".data)
else:
    raise ValueError("Only CSV and Excel files are supported.")

In [ ]:
print(df.head(5))
print(df.describe())
print(df.info())
print(df.dtypes)

#### Missing Values

In [ ]:
print("\nMISSING VALUES")
missing = df.isnull().sum()
print(missing)

if missing.sum() > 0:
    plt.figure(figsize=(8,4))
    missing[missing > 0].sort_values().plot(kind='barh')
    plt.title("Missing Values")
    plt.xlabel("Count")
    plt.show()

#### Duplicate Rows

In [ ]:
duplicates = df.duplicated().sum()
print("\nDuplicate Rows :", duplicates)


#### Unique Values

In [ ]:
print("\nUNIQUE VALUES")
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

#### Numerical Columns and Categorical Columns

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("\nNumerical Columns")
print(num_cols)

print("\nCategorical Columns")
print(cat_cols)

### Count Plot

In [ ]:
for col in cat_cols:

    print(f"\nValue Counts for {col}")
    print(df[col].value_counts())

    plt.figure(figsize=(7,4))
    sns.countplot(data=df, x=col)
    plt.xticks(rotation=45)
    plt.title(f"Count Plot - {col}")
    plt.tight_layout()
    plt.show()

#### Historgram and Box Plot

In [ ]:
for col in num_cols:

    fig, ax = plt.subplots(1,2, figsize=(12,4))

    sns.histplot(df[col], kde=True, ax=ax[0])
    ax[0].set_title(f"Histogram - {col}")

    sns.boxplot(x=df[col], ax=ax[1])
    ax[1].set_title(f"Boxplot - {col}")

    plt.tight_layout()
    plt.show()

#### Heat Map and Correlation Matrix

In [ ]:
if len(num_cols) > 1:

    plt.figure(figsize=(10,8))

    corr = df[num_cols].corr()

    sns.heatmap(corr,
                annot=True,
                cmap="coolwarm",
                fmt=".2f")

    plt.title("Correlation Matrix")
    plt.show()


#### Pair Plot

In [ ]:
if len(num_cols) <= 6:
    sns.pairplot(df[num_cols])
    plt.show()
else:
    print("\nPairplot skipped (too many numerical columns).")

#### Outlier Count IQR 

In [ ]:
print("\nOUTLIER COUNT")

for col in num_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    print(f"{col}: {outliers}")

In [ ]:
target = input("\nEnter Target Column (Leave blank if Unsupervised): ")
#target=input("\nEnter Target Column (Leave Blank if Unsupervised): ").strip()

if target == "":
    print("\nDataset Type : UNSUPERVISED")

elif target not in df.columns:
    print("Target column not found!")

else:

    y = df[target]

    # Detect Classification or Regression
    if y.dtype == object or y.nunique() <= 20:

        print("\nDetected Task : CLASSIFICATION")

        print("\nClass Distribution")
        print(y.value_counts())

        plt.figure(figsize=(7,4))
        sns.countplot(data=df, x=target)
        plt.title("Target Distribution")
        plt.show()

        # Numerical Features vs Target
        for col in num_cols:

            if col != target:

                plt.figure(figsize=(6,4))

                sns.boxplot(x=target,
                            y=col,
                            data=df)

                plt.title(f"{col} vs {target}")
                plt.tight_layout()
                plt.show()

    else:

        print("\nDetected Task : REGRESSION")

        plt.figure(figsize=(7,4))
        sns.histplot(df[target], kde=True)
        plt.title(target)
        plt.show()

        print("\nCorrelation with Target")
        print(df[num_cols].corr()[target].sort_values(ascending=False))

In [ ]:
if target in df.columns:

    if df[target].dtype == object or df[target].nunique() <= 20:

        plt.figure(figsize=(7,4))

        df[target].value_counts().plot(kind='bar')

        plt.title("Class Distribution")

        plt.ylabel("Count")

        plt.show()

#### Skewness and Kurotosis

In [ ]:
print("\nSKEWNESS")

for col in num_cols:
    print(f"{col}: {df[col].skew():.3f}")

print("\nKURTOSIS")

for col in num_cols:
    print(f"{col}: {df[col].kurtosis():.3f}")

In [ ]:
print("\n" + "="*60)
print("EDA COMPLETED SUCCESSFULLY")
print("="*60)

print(f"Rows               : {df.shape[0]}")
print(f"Columns            : {df.shape[1]}")
print(f"Missing Values     : {df.isnull().sum().sum()}")
print(f"Duplicate Rows     : {duplicates}")
print(f"Numerical Columns  : {len(num_cols)}")
print(f"Categorical Columns: {len(cat_cols)}")

## iv) Feature Selection (Only for Supervised Learning Datasets)

#### SelectKBest

In [ ]:
X = df.drop(" loan_status", axis=1)
y = df[" loan_status"]
k = 10

le = LabelEncoder()
X[" education"] = le.fit_transform(X[" education"])
X[" self_employed"] = le.fit_transform(X[" self_employed"])

# Encode the target as well
y = le.fit_transform(y)

selector = SelectKBest(score_func=f_classif, k=k)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features)

#### ANOVA

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)

X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features)

#### Column Scores ANOVA-F Test Scores

In [ ]:
import pandas as pd

scores = pd.DataFrame({
    "Feature": X.columns,
    "Score": selector.scores_
})

scores = scores.sort_values(by="Score", ascending=False)

print(scores)

## Pipeline FULL FUNCTION

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split


def ml_lab_pipeline(file_path,
                    target_column=None,
                    k=10,
                    random_state=42):

    # ==========================
    # LOAD DATASET
    # ==========================
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)

    elif file_path.endswith(".xlsx"):
        df = pd.read_excel(file_path)

    elif file_path.endswith(".data"):
        df = pd.read_csv(file_path)

    else:
        raise ValueError("Only CSV, XLSX and DATA files are supported.")

    print("="*60)
    print("DATASET LOADED")
    print("="*60)

    print(df.head())
    print(df.info())
    print(df.describe(include="all"))

    # ==========================
    # MISSING VALUES
    # ==========================
    print("\nMissing Values")
    print(df.isnull().sum())

    if df.isnull().sum().sum() > 0:
        plt.figure(figsize=(8,4))
        df.isnull().sum()[df.isnull().sum()>0].sort_values().plot(kind="barh")
        plt.title("Missing Values")
        plt.show()

    # ==========================
    # DUPLICATES
    # ==========================
    duplicates = df.duplicated().sum()
    print("\nDuplicate Rows:", duplicates)

    # ==========================
    # UNIQUE VALUES
    # ==========================
    print("\nUnique Values")

    for col in df.columns:
        print(col, ":", df[col].nunique())

    # ==========================
    # COLUMN TYPES
    # ==========================
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

    print("\nNumerical Columns")
    print(num_cols)

    print("\nCategorical Columns")
    print(cat_cols)

    # ==========================
    # COUNT PLOTS
    # ==========================
    for col in cat_cols:

        print(f"\nValue Counts of {col}")
        print(df[col].value_counts())

        plt.figure(figsize=(7,4))
        sns.countplot(data=df, x=col)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

    # ==========================
    # HISTOGRAM & BOXPLOT
    # ==========================
    for col in num_cols:

        fig, ax = plt.subplots(1,2, figsize=(12,4))

        sns.histplot(df[col], kde=True, ax=ax[0])
        ax[0].set_title(col)

        sns.boxplot(x=df[col], ax=ax[1])

        plt.tight_layout()
        plt.show()

    # ==========================
    # CORRELATION
    # ==========================
    if len(num_cols) > 1:

        plt.figure(figsize=(10,8))

        sns.heatmap(df[num_cols].corr(),
                    annot=True,
                    cmap="coolwarm",
                    fmt=".2f")

        plt.title("Correlation Matrix")
        plt.show()

    # ==========================
    # PAIRPLOT
    # ==========================
    if len(num_cols) <= 6:

        sns.pairplot(df[num_cols])
        plt.show()

    else:

        print("Pairplot skipped.")

    # ==========================
    # OUTLIERS
    # ==========================
    print("\nOutlier Count")

    for col in num_cols:

        Q1 = df[col].quantile(.25)
        Q3 = df[col].quantile(.75)

        IQR = Q3-Q1

        lower = Q1-1.5*IQR
        upper = Q3+1.5*IQR

        outliers=((df[col]<lower)|(df[col]>upper)).sum()

        print(col,":",outliers)

    # ==========================
    # SUPERVISED / UNSUPERVISED
    # ==========================
    if target_column is None:

        print("\nDataset Type : UNSUPERVISED")
        print("\nEDA Completed Successfully")

        return df

    if target_column not in df.columns:

        raise ValueError("Target column not found.")

    X=df.drop(target_column,axis=1)
    y=df[target_column]

    if y.dtype=="object" or y.nunique()<=20:
        print("\nLearning Type : SUPERVISED")
        print("Problem : Classification")
    else:
        print("\nLearning Type : SUPERVISED")
        print("Problem : Regression")

    # ==========================
    # CLASS DISTRIBUTION
    # ==========================
    if y.dtype=="object" or y.nunique()<=20:

        plt.figure(figsize=(6,4))
        y.value_counts().plot(kind="bar")
        plt.title("Class Distribution")
        plt.show()

    # ==========================
    # SKEWNESS & KURTOSIS
    # ==========================
    print("\nSkewness")

    for col in num_cols:
        print(col,":",round(df[col].skew(),3))

    print("\nKurtosis")

    for col in num_cols:
        print(col,":",round(df[col].kurtosis(),3))

    # ==========================
    # LABEL ENCODING
    # ==========================
    le=LabelEncoder()

    for col in X.select_dtypes(include="object").columns:
        X[col]=le.fit_transform(X[col].astype(str))

    if y.dtype=="object":
        y=le.fit_transform(y)

    # ==========================
    # FEATURE SELECTION
    # ==========================
    k=min(k,X.shape[1])

    selector=SelectKBest(score_func=f_classif,k=k)

    selector.fit(X,y)

    selected_features=X.columns[selector.get_support()]

    print("\nSelected Features")
    print(selected_features.tolist())

    scores=pd.DataFrame({
        "Feature":X.columns,
        "Score":selector.scores_
    })

    scores=scores.sort_values("Score",ascending=False)

    print("\nFeature Scores")
    print(scores)

    # ==========================
    # TRAIN / VALIDATION / TEST
    # ==========================
    X_train,X_temp,y_train,y_temp=train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=random_state
    )

    X_val,X_test,y_val,y_test=train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=random_state
    )

    print("\nDataset Split")
    print("Training :",X_train.shape)
    print("Validation :",X_val.shape)
    print("Testing :",X_test.shape)

    print("\nEDA Completed Successfully")

    return {
        "df":df,
        "X_train":X_train,
        "X_val":X_val,
        "X_test":X_test,
        "y_train":y_train,
        "y_val":y_val,
        "y_test":y_test,
        "selected_features":selected_features,
        "feature_scores":scores
    }

In [ ]:
#result = ml_lab_pipeline(".csv")

#result = ml_lab_pipeline(
#    "heart.csv",
#   target_column="Diagnosis",
#    k=10
#)

#result = ml_lab_pipeline(
#    "housing.csv",
#    target_column="Price",
#    k=8
#)
